### Sel 0 - Persiapan Awal (Khusus Google Colab)

Kalau notebook ini dibuka di Google Colab lewat link GitHub, jalankan sel di bawah ini dulu (atau langsung **Runtime > Run all**). Sel ini cuma memastikan semua paket yang dibutuhkan sudah tersedia. Bab ini tidak butuh file data apa pun karena dataset Breast Cancer dimuat langsung dari scikit-learn, jadi kalau dijalankan di laptop pun sel ini tidak mengubah apa-apa.

In [ ]:
# Sel 0 - Persiapan awal (jalan otomatis paling awal saat Run All).
# Bab 7 tidak membutuhkan file data eksternal: dataset Breast Cancer
# dimuat langsung dari sklearn.datasets. Sel ini hanya memastikan
# semua paket yang dibutuhkan sudah tersedia.
import importlib

for _paket in ["sklearn", "pandas", "numpy", "matplotlib"]:
    importlib.import_module(_paket)
    print(f"OK: {_paket} tersedia.")

print("Paket siap.")

# Praktikum Machine Learning: Bab 7
## Evaluasi Model Klasifikasi

| | |
|---|---|
| **Nama** | Ahmad Dandi Subhani |
| **NPM** | 202343500126 |
| **Kelas** | R7B |
| **Mata Kuliah** | Machine Learning |
| **Dosen** | Nurfidah Dwitiyanti, M.Si. |

> **Catatan:** notebook ini saya jalankan di Google Colab / Jupyter Notebook. Bab 7 ini mengerjakan **Latihan Praktikum Modul Bab 7 bagian 7.13**, memakai dataset **Breast Cancer** dari `sklearn.datasets.load_breast_cancer`. Tidak perlu file CSV apa pun.

## Ringkasan Konsep

- **Confusion matrix.** Tabel 2x2 yang merangkum hasil prediksi model: True Positive (TP), True Negative (TN), False Positive (FP), dan False Negative (FN). Semua metrik klasifikasi diturunkan dari empat angka ini.
- **Precision, recall, F1.** Precision = TP/(TP+FP), artinya seberapa tepat prediksi positif model. Recall = TP/(TP+FN), artinya seberapa banyak kasus positif yang berhasil ditangkap. F1 adalah rata-rata harmonik keduanya, enak dipakai sebagai satu angka ringkasan.
- **ROC curve dan AUC.** ROC memplot True Positive Rate melawan False Positive Rate di berbagai threshold. AUC (luas area di bawah kurva) makin dekat ke 1 berarti model makin bagus memisahkan kedua kelas.
- **PR curve.** Memplot precision melawan recall. Kurva ini lebih informatif daripada ROC kalau datanya imbalance, karena fokusnya ke kelas positif.
- **Cross-validation dan stratified k-fold.** Data dibagi jadi k lipatan, tiap lipatan gantian jadi data uji. Versi stratified menjaga proporsi kelas di tiap lipatan tetap sama seperti data aslinya, jadi hasil evaluasinya lebih andal daripada sekali split biasa.

## 7.13 Latihan Praktikum (Modul Bab 7)

Di bagian ini saya ngerjain dua latihan praktikum dari Modul Machine Learning Bab 7 bagian 7.13, semuanya memakai dataset Breast Cancer.

### Praktikum 1 - Laporan Evaluasi Lengkap

**Tujuan:** saya bisa menyusun laporan evaluasi model yang komprehensif.

**Instruksi (modul):** memakai dataset `load_breast_cancer`, latih model Logistic Regression dan Decision Tree. Buat confusion matrix, hitung precision/recall/F1 untuk keduanya, serta buat ROC curve dan PR curve dalam satu grafik untuk membandingkan kedua model.

Pertama saya siapkan dulu datanya: muat dataset Breast Cancer, lalu bagi jadi data latih dan data uji dengan `test_size=0.2` dan `random_state=42`. Setelah itu saya latih dua modelnya, Logistic Regression (`max_iter=5000`) dan Decision Tree (`random_state=42`).

In [ ]:
# Praktikum 1 - Persiapan data dan model
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

data = load_breast_cancer()
X, y = data.data, data.target
print(f"Dataset: {X.shape[0]} pasien, {X.shape[1]} fitur")
print(f"Kelas: {dict(zip(data.target_names, np.bincount(y)))}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Data latih: {X_train.shape[0]} baris, data uji: {X_test.shape[0]} baris")

model_lr = LogisticRegression(max_iter=5000)
model_dt = DecisionTreeClassifier(random_state=42)
model_lr.fit(X_train, y_train)
model_dt.fit(X_train, y_train)
print("Kedua model berhasil dilatih.")

[[PENJELASAN_1]]

Terus saya cetak `classification_report` buat kedua model supaya precision, recall, dan F1-nya kelihatan per kelas.

In [ ]:
# Praktikum 1 - Classification report kedua model
from sklearn.metrics import classification_report

for nama, model in [("Logistic Regression", model_lr), ("Decision Tree", model_dt)]:
    print(f"=== {nama} ===")
    print(classification_report(y_test, model.predict(X_test),
                                target_names=data.target_names))

[[PENJELASAN_2]]

Selanjutnya saya tampilkan confusion matrix kedua model berdampingan biar gampang dibandingkan.

In [ ]:
# Praktikum 1 - Confusion matrix berdampingan
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (nama, model) in zip(axes, [("Logistic Regression", model_lr),
                                    ("Decision Tree", model_dt)]):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test, display_labels=data.target_names,
        cmap="Blues", ax=ax)
    ax.set_title(nama)
plt.tight_layout()
plt.show()

[[PENJELASAN_3]]

Terakhir buat Praktikum 1, saya gambar ROC curve dan PR curve kedua model dalam satu figure (dua subplot berdampingan) sesuai instruksi modul.

In [ ]:
# Praktikum 1 - ROC curve dan PR curve dalam satu figure
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, auc
from sklearn.metrics import roc_curve, precision_recall_curve

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
for nama, model in [("Logistic Regression", model_lr), ("Decision Tree", model_dt)]:
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax1, name=nama)
    PrecisionRecallDisplay.from_estimator(model, X_test, y_test, ax=ax2, name=nama)
    skor = model.predict_proba(X_test)[:, 1]
    print(f"{nama}: AUC-ROC = {auc(*roc_curve(y_test, skor)[:2]):.3f}, "
          f"AUC-PR = {auc(precision_recall_curve(y_test, skor)[1], precision_recall_curve(y_test, skor)[0]):.3f}")
ax1.set_title("ROC Curve")
ax2.set_title("PR Curve")
ax1.plot([0, 1], [0, 1], "k--", label="Tebakan acak")
ax1.legend()
plt.tight_layout()
plt.show()

[[PENJELASAN_4]]

### Praktikum 2 - Perbandingan 3 Model dengan Stratified k-Fold CV

**Tujuan:** membandingkan performa 3 algoritma berbeda secara andal.

**Instruksi (modul, Praktikum 7.6):** bandingkan Logistic Regression, Decision Tree, dan KNN memakai `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` dengan `cross_val_score` dan `scoring="f1"`. Kode di bawah ini saya pakai persis dari modul, cuma saya lengkapi import yang kurang di baris paling atas.

In [ ]:
# Praktikum 7.6 - Perbandingan 3 model dengan Stratified k-Fold CV (kode modul)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

model_list = { "Logistic Regression": LogisticRegression(max_iter=5000),
    "Decision Tree": DecisionTreeClassifier(max_depth=5,
    random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),}
skf = StratifiedKFold(n_splits=5,shuffle=True,
    random_state=42)
skor_cv = {}
for nama, model in model_list.items():
    skor = cross_val_score(model,X,y,cv=skf,scoring="f1")
    skor_cv[nama] = skor
    print(f"{nama}: F1 rata-rata = {skor.mean():.3f} (+/- {skor.std():.3f})")
# Tuliskan kesimpulan model mana yang paling baik dan paling stabil.

[[PENJELASAN_5]]

Biar perbandingannya lebih gampang dibaca, saya gambar grafik batang F1 rata-rata plus standar deviasinya untuk ketiga model.

In [ ]:
# Praktikum 2 - Grafik batang F1 rata-rata +/- std
nama_model = list(skor_cv.keys())
f1_mean = [skor_cv[n].mean() for n in nama_model]
f1_std = [skor_cv[n].std() for n in nama_model]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(nama_model, f1_mean, yerr=f1_std, capsize=8,
       color=["steelblue", "orange", "green"])
ax.set_ylim(min(f1_mean) - max(f1_std) - 0.02, 1.0)
ax.set_ylabel("F1 rata-rata (5-fold CV)")
ax.set_title("Perbandingan F1 ketiga model (Stratified 5-Fold CV)")
for n, m, s in zip(nama_model, f1_mean, f1_std):
    ax.text(n, m + s + 0.002, f"{m:.3f} +/- {s:.3f}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

[[PENJELASAN_6]]

[[KESIMPULAN]]